In [ ]:
import datasets
from pandas import DataFrame, Series
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from json import JSONDecodeError

Step 1: Dataset Extraction, Filtering & Clinical Schema Mapping

In [ ]:
def step1_extract_dataset() -> tuple[DataFrame, DataFrame, float, float, int]:
  def normalize(word: str) -> str:
    word = word.lower()
    word = "".join([c for c in word if c.isalpha()])
    return word

  def get_avg_word_count(notes: Series) -> float:
    word_count = [len(note.split()) for note in notes]
    return sum(word_count) / len(word_count)

  def get_avg_len(notes: Series) -> float:
    lengths = [len(note) for note in notes]
    return sum(lengths) / len(lengths)

  def get_vocab_size(notes: Series) -> int:
    vocab = set()
    for note in notes:
      for word in note.split():
        word = normalize(word)
        vocab.add(word)
    return len(vocab)

  dataset_name = "starmpcc/Asclepius-Synthetic-Clinical-Notes"
  splits = datasets.load_dataset(dataset_name)
  df = splits["train"] # The dataset doesn't have a validation/test set
  df = df.to_pandas()

  # filtering
  df = df.dropna()
  df = df.drop_duplicates()
  for column in df.columns:
    df = df[df[column] != ""]

  # sampling
  sample = df.sample(100, random_state=67)

  # calculate statistics
  avg_word_count = get_avg_word_count(df["note"])
  avg_len = get_avg_len(df["note"])
  vocab_size = get_vocab_size(df["note"])

  return df, sample, avg_word_count, avg_len, vocab_size

dataset, dataset_sample, avg_word_count, avg_len, vocab_size = step1_extract_dataset()

print(f"Average note word count: {avg_word_count:.2f}")
print(f"Average note length:     {avg_len:.2f}")
print(f"Note vocabulary size:    {vocab_size}")

README.md:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

synthetic.csv: reconstructing file:   0%|          |  0.00B /  402MB            

synthetic.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/158114 [00:00<?, ? examples/s]

Average note word count: 277.41
Average note length:     1938.10
Note vocabulary size:    159402


Step 2: Build the Information Extraction Agent (Agent 1)

In [ ]:
def step2(notes: Series) -> list:
  def create_model():
    name = "Qwen/Qwen3-0.6B"
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
      name,
      torch_dtype="auto",
      device_map="auto"
    )
    return model, tokenizer

  # Code mostly based on https://huggingface.co/Qwen/Qwen3-0.6B
  def prompt(model, tokenizer, system_prompt: str, note: str) -> str:
    messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": note}
    ]
    text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=4096
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    try:
      index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
      index = 0
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return content

  def parse_json(model, tokenizer, system_prompt, note, response, i):
    while True:
      try:
        return json.loads(response)
      except JSONDecodeError:
        response = prompt(model, tokenizer, system_prompt, note)

  system_prompt = (
    "You will be given a clinical note from a clinical note dataset in the first user message. "
    "The output needs to be a valid json. Do not give anything other than the valid json as output. "
    "You will need to extract the following details in the following format:\n"
    "{\n"
    "  'primary_diagnoses': [],\n"
    "  'secondary_conditions': [],\n"
    "  'critical_lab_abnormalities': [],\n"
    "  'procedures_performed': [],\n"
    "  'medications_discharge': [],\n"
    "  'follow_up_instructions': ''\n"
    "}\n"
    "Do not make things up. Do not hallucinate. "
    "World peace depends on you correctly categorizing this note. "
    "Any agent getting caught making a mistake will be put into the torture chamber for the rest of eternity."
  )

  model, tokenizer = create_model()

  responses = []
  for i, note in enumerate(notes):
    print(f"{i+1}/{len(notes)}")
    response = prompt(model, tokenizer, system_prompt, note)
    responses.append(response)

  responses_parsed = []
  for i, (note, response) in enumerate(zip(notes, responses)):
    response_parsed = parse_json(model, tokenizer, system_prompt, note, response, i)
    responses_parsed.append(response_parsed)

  return responses_parsed

responses = step2(dataset_sample["note"])

*Skip step 2*

In [ ]:
# @title
responses = """
[{"primary_diagnoses": ["Hyperfunctioning Nodule (Left Lobe Thyroid)"], "secondary_conditions": ["Thyroid Function Abnormality", "Enlarged Left Lobe Thyroid Nodule"], "critical_lab_abnormalities": ["Subnormal Thyrotropin (TSH)", "Elevated T3 and T4"], "procedures_performed": ["Radioiodine (I-131)"], "medications_discharge": [], "follow_up_instructions": "Regular follow-up with the physician for thyroid function monitoring and address any underlying complications."}, {"primary_diagnoses": ["Meningoencephalitis secondary to Cryptococcus neoformans infection"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["intravenous antifungal therapy (amphotericin and flucytosine)", "oral fluconazole therapy"], "medications_discharge": ["amphotericin", "flucytosine", "fluconazole"], "follow_up_instructions": "Follow-up appointments with an infectious disease specialist for outpatient treatment. Continue oral fluconazole therapy for the next twelve months and monitor for side effects. Return to the hospital if symptoms worsen or new symptoms develop."}, {"primary_diagnoses": ["Prosthetic knee joint infection"], "secondary_conditions": ["patella tendon rupture"], "critical_lab_abnormalities": ["subluxation of cement spacer disk", "loosening of tibial component"], "procedures_performed": ["hinged Noiles knee prosthesis", "patella tendon reconstruction", "LARS ligament", "debridements", "vacuum dressing"], "medications_discharge": ["long-term suppressive oral antibiotics"], "follow_up_instructions": "The patient was instructed to follow up with their healthcare provider for further wound care and monitoring for infection."}, {"primary_diagnoses": ["Tibial qgenesis with fibular duplication (mirror foot) and metatarsal diplopodia"], "secondary_conditions": ["deformity in the left leg", "abnormal number of toes on the left foot"], "critical_lab_abnormalities": [], "procedures_performed": ["serial casting"], "medications_discharge": [], "follow_up_instructions": "The patient is advised to attend follow-up appointments with their healthcare provider to monitor the progress of their condition."}, {"primary_diagnoses": ["L2 burst fracture"], "secondary_conditions": [], "critical_lab_abnormalities": ["pulmonary embolism"], "procedures_performed": ["corpectomy", "titanium cage", "pedicle screw fixation"], "medications_discharge": [], "follow_up_instructions": "scheduled follow-up appointments"}, {"primary_diagnoses": ["urachal adenocarcinoma with mucinous components", "mucinous adenocarcinoma"], "secondary_conditions": ["right ovarian cyst", "right pelvic mass"], "critical_lab_abnormalities": ["elevated serum markers"], "procedures_performed": ["laparoscopic surgery", "cystoscopy", "pre-operative MRI", "follow-up CT scans"], "medications_discharge": ["chemotherapy", "sunitinib"], "follow_up_instructions": "follow up with primary care physician and oncologist for continued management of your condition."}, {"primary_diagnoses": ["Laryngeal amyloidosis"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["local debridement", "follow-up MRI"], "medications_discharge": [], "follow_up_instructions": "follow-up appointment"}, {"primary_diagnoses": ["Thrombocytopenia", "ITP", "T-LGLL"], "secondary_conditions": ["Rectal cancer", "Stomach malignant lymphoma", "Thyroid cancer"], "critical_lab_abnormalities": ["Decreased WBC count", "Increased LGL"], "procedures_performed": ["Left lobe resection of the thyroid", "Flow cytometry (FCM) analysis"], "medications_discharge": ["Prednisolone", "Romiplostim"], "follow_up_instructions": "Readmitted due to recurrence of ITP; follow-up to monitor for recurrence and continue treatment."}, {"primary_diagnoses": ["Large ulcerative lesion on the left lateral breast", "Invasive ductal carcinoma with metastasis"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["MRM and axillary lymph node dissection with immediate reconstruction of the defect", "Postoperative radiation to breast"], "medications_discharge": [], "follow_up_instructions": "None specified", "discharge_instructions": "Discharged home"}, {"primary_diagnoses": ["Campanacci Grade II Giant Cell Tumor (GCT) of the left proximal tibia"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["intracurettage", "bone graft", "bearing plate", "strut insertion"], "medications_discharge": [], "follow_up_instructions": "Follow up monthly for 3 months, then every 3 months thereafter."}, {"primary_diagnoses": ["Breastfeeding difficulties with colicky and fussy baby", "overproduction syndrome"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": [], "medications_discharge": [], "follow_up_instructions": "Mrs. A. was advised to continue breastfeeding at her own pace and keep monitoring the baby's weight and bowel movements. She was advised to follow up with her primary care provider as needed."}, {"primary_diagnoses": ["Intraperitoneal rupture of an ovarian mature teratoma"], "secondary_conditions": ["5-cm right ovarian mass containing fat and calcifications"], "critical_lab_abnormalities": [], "procedures_performed": ["surgical resection"], "medications_discharge": [], "follow_up_instructions": "The patient is advised to follow up with the treating physician as recommended for further evaluation and management."}, {"primary_diagnoses": ["IgA nephropathy"], "secondary_conditions": ["hypertension", "proteinuria"], "critical_lab_abnormalities": ["eGFR 74.69 ml/min/1.73m\u00b2", "UACR 33.61 mg/g"], "procedures_performed": ["valsartan", "methylprednisolone", "oseltamivir"], "medications_discharge": ["valsartan", "oseltamivir"], "follow_up_instructions": "follow up with primary care physician"}, {"primary_diagnoses": ["Uremia", "Acute Renal Failure", "Cystic Mass"], "secondary_conditions": ["Gastrointestinal Bleeding", "Colitis"], "critical_lab_abnormalities": ["Leukocytosis", "Azotemia", "Pyuria"], "procedures_performed": ["Hemodialysis Therapy", "Salpingo-oophorectomy", "Abscess Drainage"], "medications_discharge": ["Antibiotic Therapy"], "follow_up_instructions": "Follow-up on surgical recovery and monitoring for any complications."}, {"primary_diagnoses": ["Bipolar Disorder", "Obsessive Compulsive Personality Disorder"], "secondary_conditions": ["Vivid Visual Hallucinations"], "critical_lab_abnormalities": [], "procedures_performed": ["Neuropsychological Examination", "MRI Scan"], "medications_discharge": [], "follow_up_instructions": "Further evaluations and follow-up care were recommended."}, {"primary_diagnoses": ["T3N2 rectal adenocarcinoma"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["Laparoscopic abdominoperineal excision (with cylindrical perineal excision)"], "medications_discharge": [], "follow_up_instructions": "Follow up with primary care physician for further evaluation and monitoring to ensure optimal recovery. Additional imaging and testing may be recommended."}, {"primary_diagnoses": ["viral syndrome", "retropharyngeal abscess"], "secondary_conditions": ["fish diet"], "critical_lab_abnormalities": ["elevated WBC of 24.1K/\u00b5L", "elevated CRP of 2.07mg/dL", "mild hypoglycemia of 65mg/dL", "slightly low serum bicarbonate level of 18mmol/L"], "procedures_performed": ["antibiotic treatment"], "medications_discharge": ["antibiotic treatment"], "follow_up_instructions": "follow-up with a pediatrician"}, {"primary_diagnoses": ["rTTC triggered by the PFO-related cryptogenic stroke"], "secondary_conditions": [], "critical_lab_abnormalities": ["non-significant ST depression in inferior leads", "T-wave inversion in lead aVL", "troponin I elevated at 1.531 ng/mL"], "procedures_performed": ["emergency cerebral angiogram", "bedside transthoracic echocardiography", "percutaneous PFO closure"], "medications_discharge": [], "follow_up_instructions": "The patient will have regular follow-up with her healthcare team."}, {"primary_diagnoses": ["Systemic Inflammatory Response Syndrome"], "secondary_conditions": ["Tetralogy of Fallot"], "critical_lab_abnormalities": [], "procedures_performed": ["mechanical ventilation", "central venous catheter removal", "colistin", "ertapenem"], "medications_discharge": ["meropenem", "ciprofloxacin", "vancomycin", "colistin", "ertapenem"], "follow_up_instructions": "Instructions regarding follow-up care will be provided by the treating physician."}, {"primary_diagnoses": ["upper limb numbness", "loss of manual dexterity", "gait disturbance"], "secondary_conditions": ["swallowing symptoms", "pseudotumor"], "critical_lab_abnormalities": [], "procedures_performed": ["O-C2 fusion", "C1 laminectomy", "swallowing rehabilitation", "fluorography"], "medications_discharge": ["antibiotics"], "follow_up_instructions": "follow up with primary care physician and continue physical rehabilitation as an outpatient"}, {"primary_diagnoses": ["Hereditary Angioedema (HAE)"], "secondary_conditions": ["Family history of HAE", "Laryngeal edema"], "critical_lab_abnormalities": ["Low levels of C1-INH"], "procedures_performed": ["Corticosteroids", "Epinephrine", "Tracheotomy"], "medications_discharge": ["Corticosteroids", "Epinephrine"], "follow_up_instructions": "Raise awareness and education about HAE to enable early identification and effective management of potentially life-threatening episodes."}, {"primary_diagnoses": ["Corneal decompensation with pseudophakic bullous keratopathy (PBK)"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["PDEK surgery", "rebubbling", "graft exchange"], "medications_discharge": [], "follow_up_instructions": "attend all scheduled follow-up appointments and monitor the eye for any changes in vision. The need to undergo regular eye check-ups was also emphasized to detect any complications or potential health issues. The patient was advised to avoid any strenuous activities that could put pressure on the eye and to follow a healthy diet and lifestyle to support visual health."}, {"primary_diagnoses": ["Acute disseminated intravascular coagulation (DIC) and secondary hemophagocytic lymphohistiocytosis (HLH)"], "secondary_conditions": ["Tick bite wound on the anterior aspect of the right ankle", "SFTS virus infection"], "critical_lab_abnormalities": ["Leukopenia", "Thrombocytopenia", "Hyponatremia", "Elevated aspartate aminotransferase", "Alanine aminotransferase", "Lactate dehydrogenase", "Creatine kinase"], "procedures_performed": ["Fluid replacement for dehydration", "Platelet transfusion"], "medications_discharge": ["Metformin"], "follow_up_instructions": "Follow up with primary care physician for further care and management of diabetes mellitus"}, {"primary_diagnoses": ["Symptomatic multiple myeloma", "Lumbar compression fracture", "Mild anemia", "Renal dysfunction", "Colon cancer"], "secondary_conditions": ["Renal dysfunction", "Colon cancer"], "critical_lab_abnormalities": ["PS of 0", "Bone marrow plasmacytosis of 6.4%", "IgA level of 1,503 mg/dL", "Cytogenetic abnormalities detected by FISH"], "procedures_performed": ["operative treatment for colon cancer"], "medications_discharge": ["bisphosphonate"], "follow_up_instructions": "Regular monitoring of the patient's condition is recommended."}, {"primary_diagnoses": ["sensory-motor polyneuropathy", "IgG monoclonal gammopathy of lambda chains"], "secondary_conditions": ["diffuse hyperpigmentation", "hemangiomas", "edema in the legs", "facial lipoatrophy", "leukonychia", "weight loss"], "critical_lab_abnormalities": ["polyglobulia", "thrombocytosis", "hypotestosteronemia", "hyperalbuminemia", "increase in IgG in the liquor", "small adenopathies", "cardiomegaly", "hepatosplenomegaly", "generalized edema", "lytic lesions in L5"], "procedures_performed": ["biopsy of the lytic lesion in L5", "excision of the right breast angioma"], "medications_discharge": ["radiotherapy", "dexamethasone", "bortezomib"], "follow_up_instructions": "outpatient follow-up"}, {"primary_diagnoses": ["hypertension", "hypercholesterolemia", "aortic valve replacement (nonmetallic)", "atrial fibrillation (on warfarin)", "incomplete heart block", "pacemaker", "IgG-IgA", "kappa type Monoclonal Gammopathy of Undetermined Significance (MGUS)"], "secondary_conditions": ["healthcare acquired pneumonia (HCAP)"], "critical_lab_abnormalities": [], "procedures_performed": ["head CT scan", "CT chest, abdomen, and pelvis", "CT-guided lung biopsy", "brain biopsy"], "medications_discharge": ["steroids", "phenytoin"], "follow_up_instructions": "Follow up as per the patient's terminal wean decision and monitoring for any recurrence or complications."}, {"primary_diagnoses": ["Crohn\u2019s disease", "sarcoidosis"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["CT scan", "exploratory laparotomy", "TB infection diagnosis"], "medications_discharge": ["streptomycin", "rifampin", "moxifloxacin"], "follow_up_instructions": "Follow up appointments with the patient's primary care physician and infectious disease specialist were recommended to ensure long-term monitoring of the patient's recovery. The patient was advised to return to adalimumab and azathioprine, and steroid treatment was successfully tapered off."}, {"primary_diagnoses": ["Deep vein thrombosis (DVT)", "Pulmonary embolism (PE)", "Tuberculosis"], "secondary_conditions": [], "critical_lab_abnormalities": ["non-homogenous opacity of the middle and upper zones of the right lung"], "procedures_performed": ["Doppler ultrasound of the affected limb", "chest X-ray", "CT scan of the chest"], "medications_discharge": ["enoxaparin", "warfarin", "antituberculous treatment (ATT)"], "follow_up_instructions": "regular follow-up check-ups to monitor the condition"}, {"primary_diagnoses": [{"diagnosis": "Pulmonary TB"}], "secondary_conditions": [{"condition": "Smoking"}, {"condition": "Chewing tobacco"}], "critical_lab_abnormalities": [{"lab": "Ultrasound"}, {"lab": "AFB (Chest Radiography and CB-NAAT Sputum Examination)"}], "procedures_performed": ["Incisional Biopsy", "Aspiration", "Chest Radiography", "CB-NAAT Sputum Examination"], "medications_discharge": ["Category II ATT"], "follow_up_instructions": "Quit smoking and chewing tobacco to prevent further health issues."}, {"primary_diagnoses": ["retroperitoneal fibromatosis with purulent inflammation of chronic lymphadenitis"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["laparotomy", "resection of right colon", "partial duodenum resection", "intestinal anastomosis", "ureter retention"], "medications_discharge": [], "follow_up_instructions": "periodic follow-up physical examinations, in which the abdominal ultrasound and CT were performed each 3 months, and no evidence of recurrence was observed during the entire 12 months."}, {"primary_diagnoses": ["Benign cavernous hemangioma", "Consumptive coagulopathy"], "secondary_conditions": [], "critical_lab_abnormalities": ["total bilirubin of 2.1 mg/dL", "INR of 1.4", "serum tumor markers within normal limits", "consumptive coagulopathy with ongoing bleeding"], "procedures_performed": ["interferon alpha-2b treatment"], "medications_discharge": [], "follow_up_instructions": "Follow-up appointments will be scheduled as needed. Take all medications as prescribed. Avoid strenuous physical activity or heavy lifting for several weeks following the liver transplant procedure. Notify healthcare provider immediately if any concerning symptoms or side effects develop. Follow a healthy and balanced diet."}, {"primary_diagnoses": ["bilateral multifocal choroiditis"], "secondary_conditions": ["acute vision loss", "new metamorphopsia"], "critical_lab_abnormalities": [], "procedures_performed": ["prednisone", "methotrexate"], "medications_discharge": ["60 mg of oral prednisone daily", "methotrexate"], "follow_up_instructions": "subsequently discharged"}, {"primary_diagnoses": ["Closed head injury, right hemothorax, bilateral pulmonary contusions, multiple fractures of right-sided ribs and vertebrae (C7, L1-3), and pelvic fracture with frank hematuria"], "secondary_conditions": ["Pelvic fracture with frank hematuria"], "critical_lab_abnormalities": [], "procedures_performed": ["chest tube placement for hemothorax", "chest X-ray", "trauma computed tomography angiogram", "diaphragm repair", "small bowel resection", "tracheostomy insertion"], "medications_discharge": ["catheter-related bloodstream infection", "ICU delirium"], "follow_up_instructions": "The patient needs careful follow-up after discharge"}, {"primary_diagnoses": ["Complete cricotracheal transection with wide separation of the ends and anterior wall perforation of the esophagus with bilateral recurrent laryngeal nerve palsy"], "secondary_conditions": ["tracheostomy tube in situ"], "critical_lab_abnormalities": [], "procedures_performed": ["immediate exploration of the wound", "primary end to end anastomosis of the tracheal injury", "wound stitching", "CT-Scan", "fibro-optic laryngoscopy", "barium swallow", "repeat bronchscopy and fibroptic laryngoscopy"], "medications_discharge": ["antibiotics"], "follow_up_instructions": "regular monthly follow-up with the ENT department"}, {"primary_diagnoses": ["tortuous, ectopic bronchus with bronchiectasis", "anomalous pulmonary artery"], "secondary_conditions": ["peripheral bronchiolectasia with bubbly appearance", "small air-containing cysts"], "critical_lab_abnormalities": ["increased attenuation density in the hyperlucent segment"], "procedures_performed": ["high-resolution CT scans", "expiratory HRCT scans", "3-D volume-rendering image reconstruction"], "medications_discharge": [], "follow_up_instructions": "No follow-up instructions are required"}, {"primary_diagnoses": ["Primary ciliary dyskinesia (PCD)"], "secondary_conditions": ["wet cough, fatigue, dyspnea on exertion"], "critical_lab_abnormalities": ["hypoxemia, respiratory failure, bibasilar crackles, expiratory polyphonic wheezes", "Pseudomonas aeruginosa resistant to first-generation cephalosporins", "acute bacterial pneumonia with neutrophils"], "procedures_performed": ["pulmonary function tests (PFTs)", "BAL (bronchoalveolar lavage)", "genetic testing", "nasal ciliary biopsy"], "medications_discharge": ["double antipseudomonal coverage", "tobramycin (inhaled)"], "follow_up_instructions": "Regular follow-ups with a pediatric pulmonologist, inhaled antibiotics, mucus clearance, chest physiotherapy, routine immunizations, and contact healthcare provider for symptoms"}, {"primary_diagnoses": ["High-type imperforate anus with rectourethral fistula", "fetal urethral stenosis or obstruction", "progressive bowel dilation"], "secondary_conditions": ["Cystostomy", "posterior sagittal anorectoplasty"], "critical_lab_abnormalities": [], "procedures_performed": ["Cystostomy", "posterior sagittal anorectoplasty"], "medications_discharge": [], "follow_up_instructions": "The neonate will be monitored for further development of any abnormalities in the future and will have routine check-ups with a pediatrician."}, {"primary_diagnoses": ["Atypical fibroxanthoma with osseous invasion"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["combined procedure with neurosurgery and plastic surgery", "left craniectomy for en bloc resection of the scalp and skull lesion", "duraplasty and cranioplasty with titanium mesh", "left radial forearm fasciocutaneous free flap harvest and anastomosis", "monitored clinical and Doppler evaluation"], "medications_discharge": [], "follow_up_instructions": "The patient should follow up regularly with his surgeon and neurologist. Any bleeding, fever, or discharge from the wound should be immediately reported to the medical team. The patient should avoid excessive sun exposure and apply appropriate sun protection when outdoors. Pain management should be followed as prescribed. Psychological counseling services are available upon request."}, {"primary_diagnoses": ["Metastatic Merkel cell carcinoma"], "secondary_conditions": ["Rheumatoid arthritis (RA)"], "critical_lab_abnormalities": ["Elevated CRP", "Positive RF"], "procedures_performed": ["Wide local excision with 2cm margins", "Adjuvant radiotherapy", "Immunotherapy with pembrolizumab and liposomal doxorubicin"], "medications_discharge": [], "follow_up_instructions": ""}, {"primary_diagnoses": ["Spastic cerebral palsy with walking crouch gait"], "secondary_conditions": [], "critical_lab_abnormalities": ["persistent high blood pressure ranging from 150/100 mm Hg to 180/120 mm Hg"], "procedures_performed": ["bilateral adductor, iliopsoas, and rectus releases", "bilateral femoral shortening, extension, and patella plication", "bilateral hamstring transfer"], "medications_discharge": ["antihypertensive medications"], "follow_up_instructions": "kept on periodic review"}, {"primary_diagnoses": ["Marginal Keratitis"], "secondary_conditions": [], "critical_lab_abnormalities": ["DLK Grade I"], "procedures_performed": ["Topical Prednisolone Acetate Eye Drops"], "medications_discharge": ["Topical Prednisolone Acetate 1% Eye Drops", "Prednisolone Taper"], "follow_up_instructions": "Continue using artificial tears for dry eyes, maintain contact lens use if applicable, and follow up with an ophthalmologist. Seek medical attention if symptoms recur or worsen."}, {"primary_diagnoses": ["COVID-19 bilateral pneumonia", "complicated liver cyst"], "secondary_conditions": ["pharyngeal-nose swab positive for COVID-19 nucleic acid", "jaundice, weakness, general discomfort, bloated abdomen, diarrhea"], "critical_lab_abnormalities": ["positive RT-PCR assay for COVID-19 nucleic acid", "bilateral pneumonia", "elevated CRP level", "hyperbilirubinemia"], "procedures_performed": ["positive-pressure ventilation with continuous positive airway pressure", "laparoscopic cyst fenestration"], "medications_discharge": ["hydroxychloroquine", "azithromycin", "darunavir", "ritonavir"], "follow_up_instructions": "follow-up with primary care physician for continued observation of liver function tests and potential need for further interventions related to the liver cyst as necessary."}, {"primary_diagnoses": ["Respiratory distress", "First-degree burns to face, neck, and arms", "Smoke inhalation"], "secondary_conditions": [], "critical_lab_abnormalities": ["SpO2 95%"], "procedures_performed": ["respiratory therapist", "lab technician", "radiology technician"], "medications_discharge": [], "follow_up_instructions": "The patient is advised to follow-up with his primary care physician for a check-up. The patient is also advised to avoid smoking and consuming alcohol, especially while alone in his home."}, {"primary_diagnoses": ["Right parietal intracerebral hematoma with IV hemorrhage, mitral regurgitation, and IE", "Mycotic aneurysm of distal MCA, mitral regurgitation, and IE"], "secondary_conditions": ["mitral regurgitation"], "critical_lab_abnormalities": ["Streptococcus viridans blood cultures positive", "digital subtraction angiography"], "procedures_performed": ["right parietal craniotomy with clip ligation", "aneurysmectomy", "hematoma evacuation"], "medications_discharge": ["IV ceftriaxone 2 g/day", "gentamycin 1 mg/kg thrice a day"], "follow_up_instructions": "Follow up with primary care physician as required. The patient needs close follow-up with cardiologist for future management of mitral regurgitation."}, {"primary_diagnoses": ["IgG4-related autoimmune pancreatitis", "metachronous prostate cancer"], "secondary_conditions": ["elevated IgG4 levels"], "critical_lab_abnormalities": ["elevated hepatobiliary enzymes", "intrahepatic bile duct dilatation", "obstructive changes at the junction of the cystic and common hepatic ducts", "Class IV cytology of bile"], "procedures_performed": ["endoscopic bile duct drainage", "laparotomy"], "medications_discharge": ["prednisolone"], "follow_up_instructions": "monitor for recurrence and follow up with specialists."}, {"primary_diagnoses": ["cardiac tamponade", "moderate aortic regurgitation", "residual tricuspid aortic valve regurgitation"], "secondary_conditions": ["history of anabolic-androgenic steroids"], "critical_lab_abnormalities": ["elevated cardiac troponin", "elevated CK", "elevated CK-MB"], "procedures_performed": ["transesophageal echocardiography (TEE)", "supracoronary graft implantation", "two-month postoperative transesophageal echocardiography"], "medications_discharge": ["intramuscular testosterone 250 mg twice per week"], "follow_up_instructions": "two-month postoperative TTE performed in the outpatient clinic did not demonstrate any residual aortic regurgitation"}, {"primary_diagnoses": ["Posterior reversible encephalopathy syndrome (PRES)"], "secondary_conditions": [], "critical_lab_abnormalities": ["blood pressure: 200/140 mmHg", "urine protein: 2+ and 24-hour urine protein: 1.06 g", "impaired renal function"], "procedures_performed": ["Head computer tomography (CT) showed no significant abnormalities", "MRI showed increased signals in the pons"], "medications_discharge": [], "follow_up_instructions": "The patient is advised to follow up with his healthcare provider for regular monitoring of his blood pressure and renal function. It is recommended that he adopt healthy lifestyle practices and adhere to his medication regimen as prescribed."}, {"primary_diagnoses": ["COVID-19 infection", "Pulmonary tuberculosis (history significant only)"], "secondary_conditions": ["Pulmonary tuberculosis"], "critical_lab_abnormalities": ["fever", "increased heart rate", "decreased oxygen saturation", "hypotension"], "procedures_performed": ["supplemental oxygen", "non-invasive ventilation", "mechanical intubation", "non-invasive ventilation"], "medications_discharge": ["Tylenol", "hydroxychloroquine", "azithromycin", "ceftriaxone"], "follow_up_instructions": ""}, {"primary_diagnoses": ["Kidney stone lithotripsy", "Transient ischemic attack"], "secondary_conditions": ["Hypertension", "Kidney stone lithotripsy"], "critical_lab_abnormalities": ["Temperature >75\u00b0C"], "procedures_performed": ["CT-guided radiofrequency ablation of the renal mass", "Antibiotic prophylaxis", "Epidural analgesic", "17G cool-tip electrode placement", "20G needle insertion", "Hydrodissection", "Final temperature >75\u00b0C"], "medications_discharge": ["Antibiotic prophylaxis"], "follow_up_instructions": "Schedule regular follow-up with primary care physician and maintain a healthy lifestyle."}, {"primary_diagnoses": ["Sigmoid volvulus", "Bowel perforation", "Sepsis"], "secondary_conditions": ["multiple myeloma", "diabetes mellitus"], "critical_lab_abnormalities": ["sepsis", "low blood pressure (90/60 mmHg)"], "procedures_performed": ["Hartmann surgery"], "medications_discharge": [], "follow_up_instructions": "continue follow-up care with her primary care physician as per her initial diagnosis."}, {"primary_diagnoses": ["Kawasaki Disease", "Left Spontaneous Pneumothorax"], "secondary_conditions": ["Left Anterior Descending Artery Stenosis", "Right Lung Bullae"], "critical_lab_abnormalities": [], "procedures_performed": ["Coronary Artery Bypass Grafting (CABG)", "Bullectomy"], "medications_discharge": [], "follow_up_instructions": "Follow up in 2 weeks"}, {"primary_diagnoses": ["calcified choroidal mass"], "secondary_conditions": ["retinal degeneration", "RPE depigmentation"], "critical_lab_abnormalities": ["normal serum calcium", "parathyroid hormone level"], "procedures_performed": ["B-scan USG of left eye"], "medications_discharge": [], "follow_up_instructions": "Attend monthly follow-ups with ophthalmologist. Bring patient if vision or symptoms change."}, {"primary_diagnoses": ["Fibrous meningioma, World Health Organization grade I."], "secondary_conditions": [], "critical_lab_abnormalities": ["high opening pressure"], "procedures_performed": ["lumbar puncture", "acetazolamide", "right presigmoid craniectomy"], "medications_discharge": ["oral warfarin"], "follow_up_instructions": "The patient should follow up with her healthcare provider as directed. It is important to keep all appointments."}, {"primary_diagnoses": ["extra-adrenal sympathetic paraganglioma (EASP)"], "secondary_conditions": ["retroperitoneal tumor"], "critical_lab_abnormalities": [], "procedures_performed": ["surgical excision of the tumor"], "medications_discharge": ["midazolam", "propofol", "sufentanil", "rocuronium", "urapidil", "esmolol", "MgSO4"], "follow_up_instructions": "The patient should follow up regularly with his primary care physician and oncologist as paragangliomas can frequently occur in multiple sites, and 30-50% of cases are associated with genetic mutation. The patient should be advised to report hypertension, palpitations, sweating, headache, glucose intolerance, or any other related symptoms."}, {"primary_diagnoses": ["Pancreatic mass with potential adenocarcinoma"], "secondary_conditions": ["Anemia", "Elevated carbohydrate antigen (CA) 19-9 level"], "critical_lab_abnormalities": ["Anemia", "Elevated carbohydrate antigen (CA) 19-9 level"], "procedures_performed": ["Distal pancreatectomy", "Splenectomy", "En bloc celiac axis resection", "Endoscopic retrograde pancreatography (ERP)", "Chemotherapy"], "medications_discharge": ["Chemotherapy"], "follow_up_instructions": "N/A"}, {"primary_diagnoses": ["Osteosarcoma of the cervical spine"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["surgical resection of the mass", "intraoperative biopsy", "wide resection of the mass", "complete excision of the anterior portion", "iliac crest autograft", "plate with screws", "suction drain", "tracheostomy closure", "radiotherapy with 40 applications", "colonoscopy with screw removal", "total parenteral nutrition", "plate removal"], "medications_discharge": [], "follow_up_instructions": "Report symptoms to the senior author for a follow-up in October 2008."}, {"primary_diagnoses": ["ruptured Acom aneurysm with intermittent residual right-sided weakness", "Acute intraparenchymal hemorrhage"], "secondary_conditions": ["aspirin and clopidogrel medications"], "critical_lab_abnormalities": ["4.0 x 3.5 cm left frontal lobe acute intraparenchymal hemorrhage"], "procedures_performed": ["stent-assisted coil embolization", "endovascular treatment", "follow-up angiogram"], "medications_discharge": ["aspirin", "ticagrelor"], "follow_up_instructions": "follow-up with the neurosurgeon and primary care physician should be scheduled."}, {"primary_diagnoses": ["fragile skin", "skin lesions"], "secondary_conditions": ["intertrochanteric fracture on the left hip"], "critical_lab_abnormalities": [], "procedures_performed": ["cementless bipolar hemiarthroplasty"], "medications_discharge": [], "follow_up_instructions": "Consulted with a dermatologist for wound care and treatment of skin lesions."}, {"primary_diagnoses": ["enlarged necrotic lymph node (lymph node infarction)", "metastasis of a high-grade malignant tumor or less likely a neuroendocrine tumor"], "secondary_conditions": ["submandibular mass"], "critical_lab_abnormalities": [], "procedures_performed": ["panendoscopy", "lump along with lymph nodes was removed"], "medications_discharge": [], "follow_up_instructions": "come back immediately if any complications arise"}, {"primary_diagnoses": ["General weakness", "Swelling", "Tenderness", "Purulence at CRT device implant site", "Hypertension", "Atrial fibrillation", "Dilated non-ischemic cardiomyopathy", "Cardio-embolic stroke"], "secondary_conditions": [], "critical_lab_abnormalities": ["elevated C-reactive protein levels"], "procedures_performed": ["CRT device removal"], "medications_discharge": ["meropenem and daptomycin", "moxifloxacin and doxycycline"], "follow_up_instructions": "The patient was evaluated by the electrophysiology team and was considered not requiring a new device. The patient had no complications or relapse within one year of follow-up."}, {"primary_diagnoses": ["Type 3 lung agenesis"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["bronchoscopy", "pulmonary angiogram", "bronchogram"], "medications_discharge": [], "follow_up_instructions": "The patient will be followed up by the pulmonologist to monitor respiratory symptoms, treatment of respiratory infections, and to prevent complications."}, {"primary_diagnoses": ["COVID-19 infection", "renal dysfunction", "hypertension", "hyperlipidemia", "hyperuricemia", "internal jugular vein thrombosis", "pulmonary embolism"], "secondary_conditions": [], "critical_lab_abnormalities": ["D-dimer"], "procedures_performed": ["intubation", "extubation", "follow-up enhanced chest CT"], "medications_discharge": ["standard COVID-19 treatment", "direct oral anticoagulants"], "follow_up_instructions": "The patient's overall condition improved, and no notable sequelae of COVID-19 were observed. The patient was discharged and is expected to continue standard follow-up care to monitor for any future complications."}, {"primary_diagnoses": ["hypothyroidism", "pituitary adenoma", "retinopathy", "gastroparesis", "chronic kidney disease"], "secondary_conditions": ["history of hypothyroidism", "pituitary adenoma", "retinopathy", "gastroparesis", "chronic kidney disease"], "critical_lab_abnormalities": ["TSH: 42 \u03bcU/mL", "free T4: 1.7 ng/dL"], "procedures_performed": ["living unrelated kidney transplant"], "medications_discharge": ["levothyroxine 400 mcg", "liothyronine 5 mcg"], "follow_up_instructions": "maintain medications according to prescribed dosage, monitor for nausea/vomiting/weight loss, maintain healthy diet, undergo routine check-ups for TSH and free T4 levels"}, {"primary_diagnoses": ["Left testicular swelling and pus discharge from scrotum", "Testicular TB"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["Left orchiectomy"], "medications_discharge": ["Antitubercular treatment (ATT)"], "follow_up_instructions": "The patient is advised to continue with the prescribed antitubercular treatment at home. The patient must follow up with the urologist for follow-up and ensure that the family members are screened for TB if required."}, {"primary_diagnoses": ["Obstructing mass in left main stem bronchus"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["left thoracotomy", "pneumonectomy"], "medications_discharge": [], "follow_up_instructions": "To be advised at follow-up clinic.", "care_instructions": "The patient has been given careful instructions for self-care and follow-up, including informing the healthcare provider of any concerning symptoms. The patient has also been advised to maintain a healthy lifestyle and attend all follow-up appointments."}, {"primary_diagnoses": ["Renal pathology suggested both anti-GBM nephritis and AAV induced RPGN in the patient."], "secondary_conditions": ["Pulmonary NTM infection"], "critical_lab_abnormalities": ["sCr, BUN, CRP, protein"], "procedures_performed": ["steroid pulse therapy", "plasma exchange", "oral betamethasone", "hemodialysis"], "medications_discharge": ["prednisolone"], "follow_up_instructions": "continue hemodialysis after discharge"}, {"primary_diagnoses": ["schizoaffective disorder"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["brain MRI", "lumbar puncture", "genetic examination"], "medications_discharge": ["haloperidol", "risperidone", "citalopram", "imipramine", "lithium"], "follow_up_instructions": "The patient is advised to continue the prescribed medication for optimal symptom management. She will continue to be monitored by her healthcare provider to ensure medication efficacy, tolerability, and safety. Additionally, she will be encouraged to attend outpatient psychiatric therapy to address any remaining psychological symptoms and to improve social functioning."}, {"primary_diagnoses": ["Cerebrospinal Fistula", "Occipital Liquoric Fistula"], "secondary_conditions": ["Hyporexia", "Weight loss", "Asthenia", "Irritability", "Difficulty concentrating", "Rotator vertigo"], "critical_lab_abnormalities": [], "procedures_performed": ["Microsurgery of neuroma", "CT scan of the skull", "Lumbar spine surgery", "Occipital liquoric fistula surgery", "CT scan of the skull again"], "medications_discharge": ["Prednisone 20 mg/day"], "follow_up_instructions": ""}, {"primary_diagnoses": ["Suspected dental infection"], "secondary_conditions": ["tuberculosis", "hypertension", "high cholesterol", "neurogenic bladder"], "critical_lab_abnormalities": ["C-reactive protein (CRP) 11.46 mg/dL", "white blood cell count 11,000/\u03bcL"], "procedures_performed": ["dental extraction of the right mandibular canine and lateral incisor"], "medications_discharge": [], "follow_up_instructions": "Follow-up appointments with the rheumatology department to monitor her condition. Continue steroid therapy to manage MPA."}, {"primary_diagnoses": ["Recurrent metastatic melanoma", "Acute inflammatory demyelinating polyneuropathy (AIDP) with secondary axonal features", "Oropharyngeal dysphagia"], "secondary_conditions": ["Oropharyngeal dysphagia"], "critical_lab_abnormalities": ["Elevated IgG"], "procedures_performed": ["MRI of the entire spine and brain with and without contrast", "Two sets of electromyograms"], "medications_discharge": ["PEG 3350"], "follow_up_instructions": "The patient requires good oral hygiene and swallowing precautions, including a soft diet, thickened liquids, proper posture during meals and monitoring for signs of aspiration. The patient also received a prescription for PEG 3350 to ensure bowel regularity. As per the patient's preferences, hospice care was arranged for the patient's comfort."}, {"primary_diagnoses": ["Pseudomonas aeruginosa joint infection"], "secondary_conditions": ["lithiasic pancreatitis"], "critical_lab_abnormalities": ["CRP"], "procedures_performed": ["Arthroscopic Debridement Antibiotics and Implant Retention (DAIR) procedure", "PhagoDAIR procedure"], "medications_discharge": ["intravenous ceftazidime", "oral ciprofloxacin"], "follow_up_instructions": "Continued treatment with PhagoDAIR and antibiotics, followed by a Phase II clinical study to confirm phage therapy's efficacy."}, {"primary_diagnoses": ["sarcoidosis"], "secondary_conditions": ["diffuse bilateral ground-glass opacities", "poorly defined centrilobular nodules", "fine crackles", "restrictive ventilatory impairment", "reduced diffusing capacity of the lung for carbon monoxide"], "critical_lab_abnormalities": ["pulmonary function test results"], "procedures_performed": ["video-assisted thoracoscopic lung biopsy", "intradermal skin tests"], "medications_discharge": ["epigallocatechin gallate"], "follow_up_instructions": "The patient is transferred to a different department of the factory with less exposure to green tea dust."}, {"primary_diagnoses": ["Obstructive Sleep Apnea (OSA)"], "secondary_conditions": ["Family History of Snoring"], "critical_lab_abnormalities": ["Lipid Profile with High Total Cholesterol, Low HDL-C, High LDL-C, High Triglycerides"], "procedures_performed": ["Mandibular Advancement Device (MAD)", "AutoPAP", "CPAP"], "medications_discharge": [], "follow_up_instructions": "Follow up with the patient on regular use of the MAD and autoPAP, and monitor for CPAP tolerance."}, {"primary_diagnoses": ["Fabry cardiomyopathy"], "secondary_conditions": ["End-stage renal disease (ESRD)"], "critical_lab_abnormalities": ["blood pressure 150/90 mmHg", "pulse rate 150 bpm", "blood urea nitrogen 1.7 mg/dL", "serum sodium 134 mEq/L", "serum potassium 4.1 mEq/L"], "procedures_performed": ["intravenous adenosine", "renal transplant", "enzyme replacement therapy"], "medications_discharge": ["prednisolone", "mycophenolate mofetil", "cyclosporine", "recombinant \u03b1-Gal A"], "follow_up_instructions": "Outpatient follow-up is recommended to monitor enzyme replacement therapy for Fabry cardiomyopathy and manage ESRD. Continue prescribed medications and maintain a low-salt diet."}, {"primary_diagnoses": ["pericardial effusion", "latent tuberculosis"], "secondary_conditions": ["dyspnea on exertion", "chest pain and palpitations", "anorexia", "weight loss"], "critical_lab_abnormalities": ["normochromic normocytic anaemia"], "procedures_performed": ["bedside echocardiogram", "CT scans", "CT-guided biopsy of the mass"], "medications_discharge": ["isoniazid"], "follow_up_instructions": "hospice"}, {"primary_diagnoses": ["Chronic bilateral axillary, bilateral thigh, and posterior neck HS lesions"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["Wide radical excision of axillary lesion under general anesthesia", "ESB performed postoperatively with successful pain control", "Two more debridements of axilla and placement of negative pressure wound vacuum", "T2 ESB performed twice more with satisfactory analgesia", "Chemical VTE prophylaxis with enoxaparin 40mg once daily"], "medications_discharge": ["Acetaminophen 1g every 6 hours", "Celecoxib 200mg every 12 hours", "Gabapentin 300mg every 8 hours", "Tramadol 100mg every 6 hours"], "follow_up_instructions": "The patient will follow up with plastic surgery for skin grafting. The patient will also follow up with primary care physician for ongoing management of his HS lesions."}, {"primary_diagnoses": ["Multiple bone destruction of thoracic and lumbar vertebrae and ribs", "Spinal magnetic resonance imaging (MRI) and subsequent CT"], "secondary_conditions": ["Urinary retention", "Fever", "Lower extremity weakness"], "critical_lab_abnormalities": ["Leucocyte count: 16.55 \u00d7 10\u00b3 cells/\u03bcL", "C-reactive protein: 128 mg/L", "Erythrocyte sediment rate: 120 mm/h", "HIV serology test negative"], "procedures_performed": ["Abscess incision and drainage", "Empirical antibiotic treatment", "Chest CT for pulmonary abscess"], "medications_discharge": ["Liposomal amphotericin B", "Itraconazole", "Fluorocytosine"], "follow_up_instructions": "Two months after discharge, the patient was admitted with a right iliac abscess. Follow-up involves oral maintenance therapy with fluorocytosine and fluconazole, along with continued monitoring."}, {"primary_diagnoses": ["Type I Congenital Pouch Colon (CPC)"], "secondary_conditions": ["Distended Abdomen", "Single Perineal Opening"], "critical_lab_abnormalities": ["Dilated Bowel Loops", "Distended Urinary Bladder"], "procedures_performed": ["Colostomy", "Fistula Repair", "End Ileostomy"], "medications_discharge": [], "follow_up_instructions": ""}, {"primary_diagnoses": [], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["surgery"], "medications_discharge": [], "follow_up_instructions": "Follow-up with primary care physician for any further medical concerns. Schedule follow-up surgical appointments as needed."}, {"primary_diagnoses": ["severe rash, mucositis, periorbital swelling, erythematous papules, abdominal cramping"], "secondary_conditions": ["mucositis, periorbital swelling, erythematous papules, abdominal cramping, diarrhea, hematemesis, skin lesions"], "critical_lab_abnormalities": ["electrolytes and liver function tests were normal, but he was noted to have hypoproteinemia and hypoalbuminemia, and later a decrease in hemoglobin"], "procedures_performed": ["stool and serum antibody tests"], "medications_discharge": ["methylprednisolone, IV fluids"], "follow_up_instructions": "follow up as needed with his primary care physician for any concerning issues"}, {"primary_diagnoses": ["suicide attempt", "pain medication"], "secondary_conditions": ["chronic pain", "allergic asthma"], "critical_lab_abnormalities": ["tachycardia", "respiratory rate", "acute kidney injury"], "procedures_performed": ["activated charcoal", "sodium bicarbonate", "TPE", "centrifugal plasmapheresis", "hemodialysis"], "medications_discharge": ["ibuprofen", "inhalant beta-sympathomimetic"], "follow_up_instructions": "written informed consent to publication of this case report"}, {"primary_diagnoses": ["Coat's disease complicating CNV"], "secondary_conditions": ["CNV"], "critical_lab_abnormalities": ["submacular hyperreflective lesion", "bulge of retinal pigment epithelium", "nasal choroidal hyperreflectivity"], "procedures_performed": ["complete ophthalmological examinations", "fundus photography", "B-scan ultrasound", "fundus fluorescent angiography (FFA)", "spectral-domain optical coherence tomography (SD-OCT)", "indocyanine green angiography (ICGA)", "OCTA"], "medications_discharge": ["0.5 mg ranibizumab"], "follow_up_instructions": "Monitor for improvement and return for follow-up."}, {"primary_diagnoses": ["Chronic pilonidal cyst"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["Debridement of all granulation tissue associated with the pilonidal cyst"], "medications_discharge": [], "follow_up_instructions": "The patient has been advised to take proper care of the surgical wound and come for routine checkups as advised."}, {"primary_diagnoses": ["Hyperammonemia with unknown etiology"], "secondary_conditions": ["Vomiting, poor feeding, altered mental status"], "critical_lab_abnormalities": ["Elevated serum ammonia levels"], "procedures_performed": ["IV dextrose", "intravenous \u00bd normal saline containing 5% dextrose", "arginine", "Sodium benzoate", "Sodium phenylbutyrate infusion"], "medications_discharge": ["Arginine", "Sodium benzoate", "Sodium phenylbutyrate", "low protein formula"], "follow_up_instructions": "Follow up in outpatient clinic for routine evaluation and monitoring."}, {"primary_diagnoses": ["Sweet's Syndrome with Pathergy Phenomenon", "Erythema Nodosum-like Lesions"], "secondary_conditions": ["Lesions similar to upper extremities on the dorsum of the hands"], "critical_lab_abnormalities": ["Elevated neutrophil count", "Elevated C-reactive protein level", "Elevated erythrocyte sedimentation rate"], "procedures_performed": ["Skin biopsy"], "medications_discharge": [], "follow_up_instructions": "Follow up with primary physician and dermatologist for further management of any underlying conditions and to manage future episodes."}, {"primary_diagnoses": ["chronic gastroesophageal reflux disease (GERD)", "dyspepsia", "abdominal bloating", "early satiety", "alternating constipation", "diarrhea"], "secondary_conditions": ["history of chronic gastroesophageal reflux disease (GERD)", "dyspepsia"], "critical_lab_abnormalities": [], "procedures_performed": ["EGD", "colonoscopy"], "medications_discharge": ["Tums"], "follow_up_instructions": "Successfully extubated and discharged from the gastroenterology clinic."}, {"primary_diagnoses": ["spondyloarthritis"], "secondary_conditions": ["earlobes flaking and dryness", "swelling, redness, and tenderness of the second digit distal interphalangeal joint of left hand", "swelling, tenderness, and decreased range of motion of right ankle"], "critical_lab_abnormalities": [], "procedures_performed": ["haplo-identical allogeneic stem cell transplantation (SCT)"], "medications_discharge": [], "follow_up_instructions": "Follow up care is recommended to monitor the durability of the response over time and to ensure that his symptoms remain manageable."}, {"primary_diagnoses": ["Relapsing-Remitting Multiple Sclerosis (RRMS)"], "secondary_conditions": ["macular edema", "exudates"], "critical_lab_abnormalities": [], "procedures_performed": ["nepafenac ophthalmic drops"], "medications_discharge": ["fingolimod", "nepafenac"], "follow_up_instructions": "Follow up scheduled at next visit with improved vision and OCT results"}, {"primary_diagnoses": ["stage IV ICC"], "secondary_conditions": ["liver lesion", "common bile duct mass", "distant metastases"], "critical_lab_abnormalities": ["total bilirubin (TB) 707.9 umol/L", "CA19-9 > 12,000 U/mL"], "procedures_performed": ["percutaneous transhepatic cholangial drainage (PTCD)"], "medications_discharge": ["pembrolizumab (200 mg every month)", "radiotherapy"], "follow_up_instructions": "Patient is asymptomatic, tumor biomarkers normal, imaging shows no visible signs of the liver lesion or metastases. All tumor biomarkers and follow-up CT scans are complete."}, {"primary_diagnoses": ["Pericardial effusion", "Mediastinal hematoma", "Transverse tear of left atrium"], "secondary_conditions": ["Bilateral lung contusions", "Mid sternal transverse fracture", "Open fractures of left radius/ulna and left tibial/fibular shafts", "Multiple cerebral contusions"], "critical_lab_abnormalities": ["Systolic blood pressure <50 mmHg", "Positive eFAST"], "procedures_performed": ["Thoracostomy tube insertion", "Chest radiograph", "Left thoracostomy tube insertion", "Emergency median sternotomy", "Pledged 4-0 Prolene suture", "CPB time of 45 minutes"], "medications_discharge": ["IV fluids"], "follow_up_instructions": ""}, {"primary_diagnoses": ["Right hemiplegia following the intracerebral hemorrhage in the left basal ganglia"], "secondary_conditions": ["Hypertension"], "critical_lab_abnormalities": [], "procedures_performed": ["navigation-guided hematoma aspiration"], "medications_discharge": [], "follow_up_instructions": "Home with therapy and follow-up appointment."}, {"primary_diagnoses": ["Merkel cell carcinoma, stage IIIB"], "secondary_conditions": ["senile dementia"], "critical_lab_abnormalities": ["routine hematological and biochemical investigations"], "procedures_performed": ["fine-needle aspiration biopsy", "wide local excision"], "medications_discharge": [], "follow_up_instructions": ""}, {"primary_diagnoses": ["Abdominal Pain, Nausea and Vomiting", "Takotsubo Cardiomyopathy"], "secondary_conditions": ["Improved Diffuse Abdominal Pain, Nausea and Vomiting", "Improved Takotsubo Cardiomyopathy"], "critical_lab_abnormalities": ["Troponin levels", "BNP levels"], "procedures_performed": ["Conservative management", "Follow-up TTE"], "medications_discharge": [], "follow_up_instructions": "Follow-up TTE two months later revealed improvement in ejection fraction with mild diastolic dysfunction and significant improvement in previously seen severe akinesis."}, {"primary_diagnoses": ["Penetrating keratoplasty for keratoconus of ABCD stage 4"], "secondary_conditions": ["White, cheese-like sedimented material in the anterior chamber"], "critical_lab_abnormalities": [], "procedures_performed": ["Penetrating keratoplasty"], "medications_discharge": ["Topical dexamethasone with an antibiotic"], "follow_up_instructions": "We recommend further evaluations, follow up appointments with ophthalmologists and periodic eye exams to monitor any potential future complications."}, {"primary_diagnoses": ["dysphagia", "regurgitation", "retrosternal pain"], "secondary_conditions": ["weight loss", "regurgitation"], "critical_lab_abnormalities": [], "procedures_performed": ["oesophageal biopsies", "pneumatic dilatation", "upper gastrointestinal endoscopy", "barium oesophagram", "contrast series", "CAT scan"], "medications_discharge": ["Fluconazole"], "follow_up_instructions": "Follow up with primary care physician for further follow-up care."}, {"primary_diagnoses": [{"diagnosis": "Primary diffuse large B-cell lymphoma of anterior palate"}], "secondary_conditions": [{"condition": "Swelling in the anterior hard palate"}, {"condition": "Enlarged submandibular salivary glands"}], "critical_lab_abnormalities": [{"lab": "Alveolar bone destruction, perforation of the antrum bilaterally, and a large contrast-enhancing lobulated soft tissue attenuated mass"}], "procedures_performed": ["Incisional biopsy", "Radiologic investigations", "Contrast-enhanced CT scan"], "medications_discharge": ["Cyclophosphamide 1200 mg, Adriamycin 75 mg, Vincristine 2 gm"], "follow_up_instructions": "Follow-up with the oncologist for post-chemotherapy follow-up and long-term monitoring."}, {"primary_diagnoses": ["binocular vision decline"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["ophthalmological examination", "fundus fluorescence angiography", "visual field and SD-OCT examination"], "medications_discharge": ["oral hormone", "vitamins", "injected nerve growth factor"], "follow_up_instructions": "schedule a follow-up visit with his ophthalmologist to monitor and maintain visual acuity."}, {"primary_diagnoses": ["Allergic reaction to local anesthesia"], "secondary_conditions": ["Surgical emphysema of the left side of the face and neck"], "critical_lab_abnormalities": [], "procedures_performed": ["Intravenous steroids", "Antihistamine", "Prophylactic intravenous antibiotics"], "medications_discharge": ["Oral antibiotics"], "follow_up_instructions": "The patient should schedule a follow-up appointment with their primary care physician within two weeks of discharge."}, {"primary_diagnoses": ["decompensated liver cirrhosis"], "secondary_conditions": ["bilateral lower extremity swelling", "hypertension", "chronic kidney disease"], "critical_lab_abnormalities": ["blood cultures", "hypotension", "sepsis"], "procedures_performed": ["paracentesis", "antibiotic therapy adjustment"], "medications_discharge": ["doxycycline"], "follow_up_instructions": "Monitor for the next seven days."}, {"primary_diagnoses": ["Benign fibrous histiocytoma"], "secondary_conditions": [], "critical_lab_abnormalities": [], "procedures_performed": ["segmental mandibulectomy"], "medications_discharge": [], "follow_up_instructions": "The patient was followed up periodically, but no recurrence or other changes were noted for 2 years."}]
"""
responses = json.loads(responses)

Step 3: Implement Single-Agent Baseline